In [ ]:
!pip install fastapi
!pip show fastapi
!pip install uvicorn fastapi pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.9 MB/s eta 0:00:00
Name: fastapi
Version: 0.115.8
Summary: FastAPI framework, high performance, easy to learn, fast to code, ready for production
Home-page: https://github.com/fastapi/fastapi
Author: 
Author-email: =?utf-8?q?Sebasti=C3=A1n_Ram=C3=ADrez?= <tiangolo@gmail.com>
License: 
Location: /usr/local/lib/python3.11/dist-packages
Requires: pydantic, starlette, typing-extensions
Required-by: 
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/My Drive/motorcycle_emissions.csv"
df = pd.read_csv(file_path)
print(df.head())

# Selecting input features and target variables
X = df[['Gear', 'Acceleration (m/s²)', 'Fuel_Efficiency (km/l)']]
y = df[['CO_Emission (g/km)', 'CO2_Emission (g/km)', 'Smoke_Emission (units)']]

# Standardize the input features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Standardize the target variables
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

# Split into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Define an LSTM-based RNN Model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    LSTM(32, return_sequences=False),
    Dense(16, activation='relu'),
    Dense(y_train.shape[1], activation='linear')
])

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Reshape data for LSTM
X_train_reshaped = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Train the model
model.fit(X_train_reshaped, y_train, epochs=100, batch_size=8, validation_data=(X_test_reshaped, y_test))

# Save the model locally
model.save("trained_emission_model.h5")
print("Model trained and saved successfully!")

# Function to compare user input with training dataset
def compare_user_input(gear, acceleration, fuel_efficiency):
    # Transform input using the same scaler
    input_data = np.array([[gear, acceleration, fuel_efficiency]])
    input_scaled = scaler_X.transform(input_data)
    input_reshaped = input_scaled.reshape((input_scaled.shape[0], input_scaled.shape[1], 1))

    # Predict emissions
    predicted_emission = model.predict(input_reshaped)
    predicted_emission = scaler_y.inverse_transform(predicted_emission)  # Convert back to original scale

    # Get dataset mean values for comparison
    avg_values = df.select_dtypes(include=np.number).mean()

    # Calculate the difference between predicted values and dataset mean
    diff = predicted_emission.flatten() - avg_values[['CO_Emission (g/km)', 'CO2_Emission (g/km)', 'Smoke_Emission (units)']].values

    return {
        "Predicted CO_Emission (g/km)": predicted_emission[0][0],
        "Predicted CO2_Emission (g/km)": predicted_emission[0][1],
        "Predicted Smoke_Emission (units)": predicted_emission[0][2],
        "Difference in CO": diff[0],
        "Difference in CO2": diff[1],
        "Difference in Smoke Emission": diff[2]
    }

# Example usage:
gear = 3
acceleration = 2.0
fuel_efficiency = 35
result = compare_user_input(gear, acceleration, fuel_efficiency)
print(result)


Mounted at /content/drive
   Vehicle_Name  Gear  Acceleration (m/s²)  Fuel_Efficiency (km/l)  \
0  Bajaj CT 100     1                 2.30                      27   
1  Bajaj CT 100     2                 1.88                      32   
2  Bajaj CT 100     3                 1.50                      37   
3  Bajaj CT 100     4                 1.13                      42   
4  Bajaj CT 100     5                 0.80                      47   

   CO_Emission (g/km)  CO2_Emission (g/km)  Smoke_Emission (units)  
0                1.52                  113                      46  
1                1.36                  105                      44  
2                1.20                   97                      42  
3                1.05                   89                      40  
4                0.90                   82                      38  
Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 211ms/step - loss: 0.9330 - val_loss: 1.0109
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.0313 - val_loss: 0.9927
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.8872 - val_loss: 0.9715
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.9350 - val_loss: 0.9455
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.8856 - val_loss: 0.9098
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 1.0316 - val_loss: 0.8609
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.8768 - val_loss: 0.7960
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.8955 - val_loss: 0.7106
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.7265 - val_loss: 0.6044
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.6996 - val_loss: 0.4835
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.4751 - val_loss: 0.3761
Epoch 12/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.4984 - val_loss: 0.3171


Model trained and saved successfully!


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step
{'Predicted CO_Emission (g/km)': 1.494523, 'Predicted CO2_Emission (g/km)': 96.87385, 'Predicted Smoke_Emission (units)': 41.850357, 'Difference in CO': -0.15607695159912116, 'Difference in CO2': -0.3261520385742216, 'Difference in Smoke Emission': -0.1496429443359375}


In [ ]:

from fastapi import FastAPI, Request
import numpy as np
import tensorflow as tf
import joblib  # For loading scalers
import uvicorn
import nest_asyncio
import os
print(os.path.exists("trained_emission_model.h5"))
import tensorflow as tf
print(tf.__version__)  # Should match the version used while training


# Install nest_asyncio if not already installed
!pip install nest_asyncio
import tensorflow as tf
from tensorflow import keras

# Load and compile model
model = keras.models.load_model("trained_emission_model.h5")


#Replace 'mse' with tf.keras.losses.MeanSquaredError when compiling
model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())


print("Model is ready to use!")

app = FastAPI()

# Load trained ML model

model = keras.models.load_model("trained_emission_model.h5")  # Load the saved model
model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())
# Compile the model with the full path
model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())

# Load scalers -  Uncomment and adjust the file paths if you have saved your scalers.
# scaler_X = joblib.load("path/to/scaler_X.pkl")
# scaler_y = joblib.load("path/to/scaler_y.pkl")



@app.post("/sensor-data")
async def receive_sensor_data(request: Request):
    data = await request.json()

    # Extract data from ESP32
    speed = data["speed_kmh"]
    gear = data["gear"]
    braking = data["braking"]
    co2_emission = data["co2_emission_ppm"]

    # Convert braking to numerical value (0 = No, 1 = Yes)
    braking_value = 1 if braking else 0

    # Format input data for the ML model
    input_data = np.array([[gear, speed, braking_value]])
    input_scaled = scaler_X.transform(input_data)
    input_reshaped = input_scaled.reshape((1, input_scaled.shape[1], 1))

    # Predict emissions
    predicted_emission = model.predict(input_reshaped)
    predicted_emission = scaler_y.inverse_transform(predicted_emission)

    response = {
        "Predicted CO Emission (g/km)": predicted_emission[0][0],
        "Predicted CO2 Emission (g/km)": predicted_emission[0][1],
        "Predicted Smoke Emission (units)": predicted_emission[0][2]
    }

    return response

# Run API
import asyncio
import uvicorn

if __name__ == "__main__":
    # Apply nest_asyncio patch to allow nested event loops
    nest_asyncio.apply()

    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)

    # Use the existing event loop
    loop = asyncio.get_event_loop()

    # Run the server using the existing event loop
    loop.run_until_complete(server.serve())



False
2.18.0


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'trained_emission_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel


app = FastAPI()

# Predefined optimal CO₂ emission levels for vehicle models (ppm)
OPTIMAL_CO2_LEVELS = {
    "ModelA": 450,
    "ModelB": 500,
    "ModelC": 550
}

class EmissionData(BaseModel):
    vehicle_model: str
    co2_emission: int

@app.post("/check-emission")
def check_emission(data: EmissionData):
    model = data.vehicle_model
    co2_emission = data.co2_emission

    if model in OPTIMAL_CO2_LEVELS:
        optimal_level = OPTIMAL_CO2_LEVELS[model]
        if co2_emission > optimal_level:
            return {
                "message": f"Warning: Your CO₂ emission ({co2_emission} ppm) exceeds the optimal level ({optimal_level} ppm) for {model}!"
            }
        else:
            return {
                "message": f"Good job! Your CO₂ emission ({co2_emission} ppm) is within the optimal range for {model}."
            }
    else:
        return {
            "message": "Unknown vehicle model. Please provide a valid model."
        }

if __name__ == "__main__":
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="info", reload=True)



INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [1268] using StatReload
